
# Microsoft Sentinel Data Lake Notebook
## Summarizing CommonSecurity_ID Logs

This notebook:
- Reads **CommonSecurity_ID** from Sentinel Data Lake
- Produces an identity‑centric daily security summary
- Writes results to **CommonSecurity_ID_SPRK**

Designed for **Microsoft Sentinel notebook runtime (Spark)**.


In [ ]:

# Parameters
workspace_name = "NinjaSOC" #Replace with your Workspace Name
source_table = "CommonSecurity_ID_KQL_CL" #Replace with your Source Table Name
output_table = "CommonSecurity_ID_SPRK" #Replace with your Output Table Name  - make sure to have _SPRK suffix

lookback_days = 30


In [ ]:

from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F

provider = MicrosoftSentinelProvider(spark)

df = provider.read_table(source_table, workspace_name)
print(df.columns)

# Time filter
if "TimeGenerated" in df.columns:
    df = df.where(F.col("TimeGenerated") >= F.date_sub(F.current_timestamp(), lookback_days))


In [ ]:

# Normalize identity, device, and IP fields

df = (
    df
    .withColumn("Day", F.to_date("TimeGenerated"))
    .withColumn("User", F.coalesce("SourceUserName"))
    .withColumn("SrcIP", F.coalesce("SourceIP"))
    .withColumn("Device", F.coalesce("DestinationHostName"))
)


In [ ]:

# Security‑focused summary

summary_df = (
    df.groupBy(
        "Day",
        "User",
        "SrcIP",
        "Device",
    )
    .agg(
        F.count("*").alias("EventCount"),
        F.countDistinct("User").alias("DistinctUsers"),
        F.countDistinct("SrcIP").alias("DistinctSourceIPs"),
        F.countDistinct("Device").alias("DistinctDevices")
    )
    .orderBy(F.col("Day").desc(), F.col("EventCount").desc())
)

summary_df.show(50, truncate=False)


In [ ]:

# Write summary to new Data Lake table

rows = summary_df.count()
print(f"Rows to write: {rows}")

if rows > 0:
    provider.save_as_table(summary_df, output_table, "System tables", workspace_name)
    print(f"✅ Written to {output_table}")


In [ ]:

# Validation read
out_df = provider.read_table(output_table, "System tables")
out_df.orderBy(F.col("Day").desc()).show(50, truncate=False)
